# New dataset split


## Notebook settings

In [ ]:
from pathlib import Path
import random

source = Path("data_original/chest_xray/chest_xray")
destination = Path("data_nouveau_split")

random_state = 42
proportions = {"train": 0.70, "val": 0.15, "test": 0.15}
classes = ("NORMAL", "PNEUMONIA")

## Examination of the original dataset

In [ ]:
images = []

for ensemble in ("train", "val", "test"):

    for classe in classes:

        dossier = source / ensemble / classe
        nombre = 0

        for fichier in sorted(dossier.iterdir()): #fixer l'ordre (alphabetique)
            if fichier.is_file() and fichier.suffix.lower() in (".jpeg", ".jpg", ".png"): #on verifie qu'on est sur une image
                images.append({"chemin": fichier, "classe": classe, "ancien_split": ensemble})
                nombre += 1

        print(ensemble, classe,":", nombre,"images")

print("Total :", len(images),"images")

## Definition of groups

The dataset contains approximately 5,000 X-rays from 3,000 individuals; some X-rays belong to the same person. Since these images share identical characteristics, it would be inappropriate to separate multiple X-rays from the same individual across the training, testing, or validation sets, as this would result in data leakage. I have therefore decided to re-process all the images in the dataset and create a new split that accounts for this constraint. To determine which individual an X-ray belongs to, one must examine the naming convention. I proceeded based on the following logical assumption: for "normal" X-rays, the number XXXX in the identifier "IM-XXXX" corresponds to a specific individual. For "pneumonia" X-rays, the term "personXXXX" appears within the filename. In the following, the term "group" refers to a set of images that likely originate from the same individual.

In [ ]:
def person(fichier, classe):
    nom = fichier.stem  

    if classe == "NORMAL":
        pieces = nom.split("-")
        if pieces[0] == "NORMAL2":
            return "NORMAL::NORMAL2-IM-" + pieces[2]
        else:
            return "NORMAL::IM-" + pieces[1] #car il y a 2 types de noms pour les images normales

    if classe == "PNEUMONIA":
        pieces = nom.split("_")    
        return "PNEUMONIA::" + pieces[0] + "_" + pieces[1]
        
for image in images:
    image["groupe"] = person(image["chemin"], image["classe"])

## Identifying duplicates


In the following code, we combine groups of duplicates that are each located in a different group; this is a precautionary measure.

In [ ]:
import hashlib

images_uniques = {}
doublons = []

for img in images:
    
    empreinte = hashlib.sha256(img["chemin"].read_bytes()).hexdigest()
    img["empreinte"] = empreinte

    if empreinte not in images_uniques:
        images_uniques[empreinte] = img
    else:
        precedente = images_uniques[empreinte]
        ancien_groupe = img["groupe"]
        nouveau_groupe = precedente["groupe"]
        if ancien_groupe != nouveau_groupe:
            for autre_img in images:
                if autre_img["groupe"] == ancien_groupe:
                    autre_img["groupe"] = nouveau_groupe
        doublons.append(img)

images = list(images_uniques.values())
print("Doublons :", len(doublons))
print("Images conservées :", len(images))

## Counting the number of groups and verifying the consistency of the dataset

In [ ]:
groupes = set()
for image in images:
    groupes.add(image["groupe"])
print(len(groupes))

for grp in groupes:
    class_grp = set()
    for image in images:
        if image["groupe"] == grp:
            class_grp.add(image["classe"])
    if len(class_grp) > 1:
        raise ValueError("Même groupe contient plusieurs classes : " + grp)


This code counts the total number of groups in the dataset and returns an error if any of the groups contain images with different classes (since a single person cannot be both healthy and affected at the same time).

## Distribution of the different groups to respect the proportions of train/test/val

In [ ]:
group_mix = random.Random(random_state).sample(sorted(groupes), len(groupes))

data_train = []
data_val = []
data_test = []

for grp in group_mix:
    images_groupe = [image for image in images if image["groupe"] == grp]
    if len(data_train) < len(images) * proportions["train"]:
        data_train.extend(images_groupe)
    elif len(data_val) < len(images) * proportions["val"]:
        data_val.extend(images_groupe)
    else:
        data_test.extend(images_groupe)

In [ ]:
for nom, donnees in (("train", data_train), ("val", data_val), ("test", data_test)):
    print(f"\n{nom} : {len(donnees)} images")
    for classe in classes:
        nombre = sum(image["classe"] == classe for image in donnees)
        print(f"{classe} : {nombre} ({nombre / len(donnees):.1%})")

We verify that the proportions of "normal" and "pneumonia" cases are roughly the same across the three sets: train, validation, and test.

## Creation of the new split

In [ ]:
import shutil

destination
destination.mkdir(exist_ok=False)

for nom, donnees in (("train", data_train), ("val", data_val), ("test", data_test)):
    for classe in classes:
        (destination / nom / classe).mkdir(parents=True)
    for image in donnees:
        fichier = image["chemin"]
        nouveau_nom = image["ancien_split"] + "__" + fichier.name
        shutil.copy2(fichier, destination / nom / image["classe"] / nouveau_nom)